#### **Default Values and `Field()` in Pydantic**

Pydantic lets you define:

1. **Default values** — what a field gets when the user doesn't provide it.
2. **`Field()`** — additional configuration, validation rules, metadata and defaults for a field.

---
**Default Values:** A default value is used when a field is not provided.

In [1]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int = 23

user = User(name='Shourov Roy')
print(user.age)

23


Because `age` wasn't provided, Pydantic uses the default value `23`.

### Providing the value

```python
user = User(name="Shourov", age=25)
print(user)
```
The provided value replaces the default.

---
#### **What is `Field()`?**

`Field()` allows you to configure a Pydantic field. Import it:

```python
from pydantic import BaseModel, Field
```

Basic example:

```python
class User(BaseModel):
    name: str = Field(default="Unknown")
```

This is similar to:

```python
class User(BaseModel):
    name: str = "Unknown"
```

But `Field()` gives you many additional options.

---
**`Field()` with Validation Constraints:** For example, you can specify that an integer must be at least 18.

```python
from pydantic import BaseModel, Field

class User(BaseModel):
    age: int = Field(ge=18)
```

Here `ge = greater than or equal to`. So `user = User(age=25)` works. But user = User(age=15)` produces a validation error.

---

**Common `Field()` Constraints:**

* `ge` — greater than or equal to
* `gt` — greater than
* `le` — less than or equal to
* `lt` — less than

---

**String Length with `Field()`:** You can restrict the length of a string.

```python
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(min_length=3, max_length=20)
```

Now `user = User(username="Shourov")` is valid. But `user = User(username="ab")` is invalid because the username must contain at least 3 characters.

---
**`Field()` with a Default Value:** You can combine a default value with constraints.

```python
class User(BaseModel):
    age: int = Field(default=18, ge=18, le=100)
```

---
**`Field()` with Description:** `Field()` can also add metadata.

```python
class User(BaseModel):
    name: str = Field(
        description="The user's full name"
    )
```

This is particularly useful when using Pydantic with **FastAPI** because the description can appear in automatically generated API documentation.

---

#### **Example: A Real User Model**

In [2]:
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(
        min_length=3,
        max_length=20,
        description="User's username"
    )
    age: int = Field(
        default=18,
        ge=18,
        le=100
    )
    email: str
    active: bool = True

user = User(
    username="Shourov Roy",
    email="shourov@example.com"
)
print(user)

username='Shourov Roy' age=18 email='shourov@example.com' active=True


---
#### **Mutable Defaults**

Be careful with mutable objects such as lists and dictionaries. For example:

```python
class Student(BaseModel):
    subjects: list[str] = []
```

Pydantic handles mutable defaults safely by creating separate values for model instances, but using a `default_factory` is often clearer and especially useful when the default needs to be generated dynamically.

In [3]:
from pydantic import BaseModel

class Student(BaseModel):
    subjects: list[str] = Field(default_factory=list)

st1 = Student()
st2 = Student()

st1.subjects.append('Python')
print(st1)
print(st2)

subjects=['Python']
subjects=[]


---

**`default_factory`:** `default_factory` is useful when the default should be generated. For example:

In [7]:
from uuid import uuid4
from pydantic import BaseModel, Field 

# Every new user get a newly genenrated id
class User(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid4()))

user1 = User()
user2 = User()
print(user1)
print(user2)

id='5eb6de64-796e-4c06-8ae1-9c237dc5febb'
id='0dd657f2-506e-44da-a758-2c1bce6b15e7'


---
**`pattern`:** `pattern` is used when a string needs to follow a specific **regular expression (regex)** pattern. Example:

```python
from pydantic import BaseModel, Field

class User(BaseModel):
    username: str = Field(
        pattern=r"^[a-zA-Z0-9_]+$"
    )
```

This allows:

```text
shourov
shourov123
shourov_123
```

But rejects strings containing characters outside the pattern, such as:

```text
shourov@123
shourov roy
```

#### **Understanding the regex**

```text
^[a-zA-Z0-9_]+$
```

| Part           | Meaning                      |
| -------------- | ---------------------------- |
| `^`            | Start of string              |
| `[a-zA-Z0-9_]` | Letters, numbers, underscore |
| `+`            | One or more characters       |
| `$`            | End of string                |

---

** Email-like Pattern Example:** You can use a regex for simple formats:

```python
class User(BaseModel):
    username: str = Field(
        pattern=r"^[a-zA-Z0-9_]+$"
    )
```

For email addresses, however, it is generally better to use Pydantic's specialized types rather than trying to build a complete email regex yourself. For example, Pydantic provides:

```python
from pydantic import BaseModel
from pydantic import EmailStr

class User(BaseModel):
    email: EmailStr
```

This is cleaner than creating your own complicated email pattern.

---
**`multiple_of`:** You can require a number to be a multiple of another number.

```python
class Product(BaseModel):
    quantity: int = Field(multiple_of=5)
```

Valid:

```python
Product(quantity=5)
Product(quantity=10)
Product(quantity=25)
```

Invalid: `Product(quantity=7)` Because `7 % 5 != 0`. This can be useful for things like:

* Pack sizes
* Step values
* Fixed increments

---
**`max_digits` and `decimal_places`:** For `Decimal` values, Pydantic also supports constraints such as:

```python
from decimal import Decimal
from pydantic import BaseModel, Field

class Product(BaseModel):
    price: Decimal = Field(
        max_digits=10,
        decimal_places=2
    )
```

This can be useful for monetary values. For example `12345678.90` can fit within appropriate digit/decimal constraints.

---
**`alias`:** An alias allows a Pydantic field to have a **different external name**. For example, your Python code might use: `first_name` but incoming JSON might contain:

```json
{
    "firstName": "Shourov"
}
```

You can define:

```python
from pydantic import BaseModel, Field

class User(BaseModel):
    first_name: str = Field(alias="firstName")
```

Now:

```python
user = User(firstName="Shourov")
print(user.first_name)
```

---
**`validation_alias`:** Pydantic also supports `validation_alias`. This lets you specify a name that should be accepted **when validating input**.

```python
from pydantic import BaseModel, Field

class User(BaseModel):
    first_name: str = Field(
        validation_alias="firstName"
    )
```

Input:

```python
user = User(firstName="Shourov")
```

Then:

```python
print(user.first_name)
```

---
**`serialization_alias`:** `serialization_alias` controls the name used when the model is serialized. This is useful when your internal Python naming convention differs from your API's JSON naming convention.

---

#### **Quick Reference**

| Constraint            | Purpose                          | Example                           |
| --------------------- | -------------------------------- | --------------------------------- |
| `min_length`          | Minimum string/collection length | `min_length=3`                    |
| `max_length`          | Maximum string/collection length | `max_length=20`                   |
| `pattern`             | String regex pattern             | `pattern=r"^[A-Z]+$"`             |
| `gt`                  | Greater than                     | `gt=0`                            |
| `ge`                  | Greater/equal                    | `ge=18`                           |
| `lt`                  | Less than                        | `lt=100`                          |
| `le`                  | Less/equal                       | `le=100`                          |
| `multiple_of`         | Number must be a multiple        | `multiple_of=5`                   |
| `max_digits`          | Maximum decimal digits           | `max_digits=10`                   |
| `decimal_places`      | Maximum decimal places           | `decimal_places=2`                |
| `alias`               | External field name              | `alias="firstName"`               |
| `validation_alias`    | Input field name                 | `validation_alias="firstName"`    |
| `serialization_alias` | Output field name                | `serialization_alias="firstName"` |
| `default_factory`     | Dynamically generate default     | `default_factory=list`            |

---